# Population based model
## Link activity avoided back to original data

This notebook uses Databricks to link the activity avoided saved from each
Monte Carlo simulation back to the original data, aggregating the inpatients activity
avoided by HRG, an indicator on whether the length of stay from admission to discharge
is 0, 1 or 2+, and POD.

We also produce this aggregation for the baseline data, and for the actual modelled results.

For this notebook to work we first need to have run the national model run
from nhp_model, with save_full_model_results set to TRUE for inpatients.

This notebook currently only works for IP, not OP or AAE.

## Setup

In [0]:
import sys

sys.path.append(spark.conf.get("bundle.sourcePath", "."))
import json
import os
import pandas as pd
import pyspark.sql.functions as F
from nhpy import process_data

%load_ext autoreload
%autoreload 2

In [0]:
# Load the path to the results to use
dbutils.widgets.text("results_folder", "", "Path to Full Model Results Folder")

In [0]:
# Load and set variables

results_folder = dbutils.widgets.get("results_folder")
file_path = f"/Volumes/nhp/results/files/{results_folder}"
aggregated_path = file_path.replace("full-model-results", "aggregated-model-results")
with open(f"{aggregated_path}/params.json", "rb") as f:
    params = json.load(f)
scenario_name = params["scenario"]
seed = params["seed"]

# Check that there hasn't been a patch release of data - this is very rare
data_version = results_folder.split("/")[1] + ".0"
data_path = f"/Volumes/nhp/model_data/files/{data_version}"

## Generate baseline

In [0]:
apc_baseline = (
            spark.read.parquet(f"{data_path}/ip")
            .filter(F.col("fyear") == params["start_year"])
            .withColumn("dataset", F.lit("NATIONAL"))
            .withColumn("sitetret", F.lit("NATIONAL"))
            .sample(fraction=0.01, seed=seed)
            .persist()
)

In [0]:
apc_baseline = apc_baseline.withColumn(
    "los_group",
    F.when(F.col("speldur") == 0, F.lit("0"))
     .when(F.col("speldur") == 1, F.lit("1"))
     .when(F.col("speldur") >= 2, F.lit("2+"))
     .otherwise(F.lit(None))
).withColumn("beddays", F.col("speldur") + 1)
apc_baseline_agg = apc_baseline.groupBy("los_group", "pod", "sushrg") \
    .agg(
        F.sum("beddays").alias("beddays"),
        F.count("*").alias("admissions")
    ).orderBy(
    "los_group", "pod", "sushrg"
)
apc_baseline_agg.toPandas().to_csv(f"{scenario_name}_baseline.csv", index=False)

## Model results

In [0]:
from pyspark.sql import types as T


def add_missing_groupings(
    final_groupings_per_run,
    required_groupings,
    grouping_cols,
    value_cols=("beddays", "admissions"),
):
    """
    Efficiently add missing grouping combinations.

    Avoids:
    - giant OR expressions
    - massive cross joins
    - stack overflow issues
    """

    spark = final_groupings_per_run.sparkSession

    join_cols = ["model_run"] + grouping_cols

    # Small dimension table
    required_df = spark.createDataFrame(
        required_groupings,
        grouping_cols,
    )

    # Broadcast if reasonably small
    required_df = F.broadcast(required_df)

    # Existing combinations only
    existing_keys = (
        final_groupings_per_run
        .select(*join_cols)
        .distinct()
    )

    # All model runs
    model_runs = (
        final_groupings_per_run
        .select("model_run")
        .distinct()
    )

    # Generate candidate combinations
    candidates = (
        model_runs
        .crossJoin(required_df)
    )

    # ONLY missing combinations
    missing = (
        candidates.join(
            existing_keys,
            on=join_cols,
            how="left_anti",
        )
    )

    # Add zero-filled metric columns
    for col_name in value_cols:
        missing = missing.withColumn(
            col_name,
            F.lit(0),
        )

    # Match schema exactly
    missing = missing.select(
        *final_groupings_per_run.columns
    )

    # Append only missing rows
    final_df = final_groupings_per_run.unionByName(
        missing
    )

    return final_df

In [0]:
apc = apc_baseline.drop("speldur", "classpat")
results = spark.read.parquet(f"{file_path}/ip/")

full_results = apc.join(results, on="rn", how="left")
full_results = full_results.withColumn(
    "los_group",
    F.when(F.col("speldur") == 0, F.lit("0"))
    .when(F.col("speldur") == 1, F.lit("1"))
    .when(F.col("speldur") >= 2, F.lit("2+"))
    .otherwise(F.lit(None)),
).withColumn("beddays", F.col("speldur") + 1)
aggregated_full_results = full_results.groupBy(
    "model_run", "los_group", "pod", "sushrg"
).agg(F.sum("beddays").alias("beddays"), F.count("*").alias("admissions"))
# add missing groupings
groupings_list = [
    tuple(row)
    for row in aggregated_full_results.select("los_group", "pod", "sushrg")
    .distinct()
    .collect()
]
filled_aggregated_full_results = add_missing_groupings(
    final_groupings_per_run=aggregated_full_results,
    required_groupings=groupings_list,
    grouping_cols=["los_group", "pod", "sushrg"],
    value_cols=["beddays", "admissions"],
)
final_full_results = filled_aggregated_full_results.groupby(
    "los_group", "pod", "sushrg"
).agg(F.mean("beddays").alias("beddays"), F.mean("admissions").alias("admissions"))

final_full_results.orderBy(
    "los_group", "pod", "sushrg"
).toPandas().to_csv(f"{scenario_name}_results.csv", index=False)

## QA

In [0]:
from nhpy.process_results import convert_results_format

default = spark.read.parquet(f"{aggregated_path}/default.parquet")
df = convert_results_format(default.toPandas())
default_admissions = df[(df["pod"].str.startswith("ip_")) & (df["measure"] == "admissions")]["mean"].sum()
default_beddays = df[(df["pod"].str.startswith("ip_")) & (df["measure"] == "beddays")]["mean"].sum()

In [0]:
agg_row = final_full_results.agg(
    F.sum("admissions").alias("total_admissions"),
    F.sum("beddays").alias("total_beddays")
).collect()[0]
agg_admissions = agg_row["total_admissions"]
agg_beddays = agg_row["total_beddays"]

In [0]:
assert agg_admissions == default_admissions
assert agg_beddays == default_beddays

## Avoided activity

In [0]:
apc = apc_baseline.drop("speldur", "classpat", "pod")
avoided = spark.read.parquet(f"{file_path}/ip_avoided/")

full_avoided = avoided.join(apc, on="rn", how="left")
full_avoided = (
    full_avoided.withColumn(
        "los_group",
        F.when(F.col("speldur") == 0, F.lit("0"))
        .when(F.col("speldur") == 1, F.lit("1"))
        .when(F.col("speldur") >= 2, F.lit("2+"))
        .otherwise(F.lit(None)),
    )
    .withColumn("beddays", F.col("speldur") + 1)
    .withColumn(
        "pod",
        F.when(F.col("classpat") == "2", "ip_elective_daycase")
        .when(F.col("classpat") == "3", "ip_regular_day_attender")
        .when(F.col("classpat") == "4", "ip_regular_night_attender")
        .otherwise(F.concat(F.lit("ip_"), F.col("group"), F.lit("_admission"))),
    ) # sometimes pods change
)

In [0]:
aggregated_full_avoided = full_avoided.groupBy(
    "model_run", "los_group", "pod", "sushrg"
).agg(F.sum("beddays").alias("beddays"), F.count("*").alias("admissions"))
# add missing model runs
groupings_list = [tuple(row) for row in aggregated_full_avoided.select("los_group", "pod", "sushrg").distinct().collect()]
filled_aggregated_full_avoided = add_missing_groupings(
    final_groupings_per_run=aggregated_full_avoided,
    required_groupings=groupings_list,
    grouping_cols=["los_group", "pod", "sushrg"],
    value_cols=["beddays", "admissions"],
)
final_avoided = filled_aggregated_full_avoided.groupby(
    "los_group", "pod", "sushrg"
).agg(F.mean("beddays").alias("beddays"), F.mean("admissions").alias("admissions"))
final_avoided.orderBy(
    "los_group", "pod", "sushrg"
).toPandas().to_csv(f"{scenario_name}_activity_avoided.csv", index=False)